In [6]:
import json, os
import numpy as np
import pandas as pd
import joblib

df = pd.read_csv("../data/processed/features_v2.csv", parse_dates=["date"])
clubs = pd.read_csv("../data/raw/clubs.csv", usecols=["club_id", "name", "last_season"]).rename(
    columns={"club_id": "current_club_id", "name": "club_name"})
bundle = joblib.load("../models/lightgbm_final.joblib")
cat_cols = bundle["cat_cols"]

DATA_END = df["date"].max()
print("Data runs until:", DATA_END.date())

Data runs until: 2026-06-12


In [7]:
# Taking each club's most recent row, because it holds the club's latest 12-month average value
df = df.sort_values("date")
latest = df.dropna(subset=["club_avg_lv_prev"]).groupby("current_club_id").tail(1)

# The league table: whether performance data exists there, and its latest league-wide average
league_info = (df.groupby("player_club_domestic_competition_id")
                 .agg(covered=("league_covered", "first"), avg_lv=("league_avg_lv_prev", "last"))
                 .dropna())

# Keeping clubs seen in the last year, dropping the pooled "UNK" group (free agents, unlisted clubs)
latest = latest[
    (latest["date"] >= DATA_END - pd.Timedelta(days=365))
    & (latest["player_club_domestic_competition_id"].isin(league_info.index))
    & (latest["player_club_domestic_competition_id"] != "UNK")
]
latest = latest.merge(clubs, on="current_club_id", how="inner")  # the inner join drops clubs with no name on record
# Keeping only clubs seen in the newest season, since clubs that dropped out of the top flights keep an old league label
current_season = latest["last_season"].max()
latest = latest[latest["last_season"] == current_season]
print("Season used:", current_season)
print(latest["player_club_domestic_competition_id"].value_counts().head(12))
print(len(latest), "clubs in the dropdown")

Season used: 2025
player_club_domestic_competition_id
ARG1    30
MLS1    30
COL1    20
BRA1    20
IT1     20
GB1     20
JAP1    20
ES1     20
MEX1    18
TR1     18
L1      18
NL1     18
Name: count, dtype: int64
548 clubs in the dropdown


In [8]:
# Rebuilding the exact category lists used to train the final model (train + validation rows)
trainval = df[df["date"] < "2025-01-01"]
categories = {c: sorted(trainval[c].dropna().unique().tolist()) for c in cat_cols}

# Positions and their sub-positions for linked dropdowns; "Missing" is a data gap, not something to offer users
pos = trainval[trainval["position"] != "Missing"].dropna(subset=["position", "sub_position"])
positions = {p: sorted(g["sub_position"].unique().tolist()) for p, g in pos.groupby("position")}

# Typical height per sub-position, used as a default in the form
default_height = {k: int(v) for k, v in
                  pos.groupby("sub_position")["height_in_cm"].median().fillna(182).round().items()}

print(positions)

{'Attack': ['Centre-Forward', 'Left Winger', 'Right Winger', 'Second Striker'], 'Defender': ['Centre-Back', 'Left-Back', 'Right-Back'], 'Goalkeeper': ['Goalkeeper'], 'Midfield': ['Attacking Midfield', 'Central Midfield', 'Defensive Midfield', 'Left Midfield', 'Right Midfield']}


In [9]:
# Readable names for the 14 leagues that have performance data; other leagues just show their code
LEAGUE_NAMES = {
    "GB1": "Premier League", "ES1": "La Liga", "IT1": "Serie A", "L1": "Bundesliga",
    "FR1": "Ligue 1", "NL1": "Eredivisie", "PO1": "Liga Portugal", "TR1": "Süper Lig",
    "BE1": "Belgian Pro League", "SC1": "Scottish Premiership", "RU1": "Russian Premier League",
    "UKR1": "Ukrainian Premier League", "GR1": "Super League Greece", "DK1": "Danish Superliga",
}

lookup = {
    "data_as_of": str(DATA_END.date()),
    "clubs": [
        {"id": int(r.current_club_id), "name": r.club_name,
         "league": r.player_club_domestic_competition_id,
         "club_avg_lv": round(float(r.club_avg_lv_prev), 4),
         "club_n_prev": int(r.club_n_prev)}
        for r in latest.sort_values("club_name").itertuples()
    ],
    "leagues": {
        code: {"name": LEAGUE_NAMES.get(code, code), "covered": int(r.covered),
               "avg_lv": round(float(r.avg_lv), 4)}
        for code, r in league_info.iterrows()
    },
    "categories": categories,
    "positions": positions,
    "default_height": default_height,
}

os.makedirs("../app", exist_ok=True)
with open("../app/lookup.json", "w", encoding="utf-8") as f:
    json.dump(lookup, f, ensure_ascii=False, indent=1)
print("Saved:", round(os.path.getsize("../app/lookup.json") / 1e3, 1), "KB")

Saved: 79.7 KB


In [10]:
chk = pd.DataFrame(lookup["clubs"])
# Converting the log average back to euros; this is a "typical" value, not a plain mean
chk["typical_value_eur"] = np.expm1(chk["club_avg_lv"]).round(-3)

print(chk.nlargest(8, "typical_value_eur")[["name", "league", "typical_value_eur"]].to_string(index=False))
print()
print(chk.nsmallest(5, "typical_value_eur")[["name", "league", "typical_value_eur"]].to_string(index=False))
print()
print("Clubs per league (top 10):")
print(chk["league"].value_counts().head(10))

               name league  typical_value_eur
         Arsenal FC    GB1         45308000.0
    Manchester City    GB1         32874000.0
         Chelsea FC    GB1         31823000.0
Paris Saint-Germain    FR1         29239000.0
        Real Madrid    ES1         27840000.0
       Liverpool FC    GB1         26480000.0
  Manchester United    GB1         25582000.0
       FC Barcelona    ES1         25472000.0

                           name league  typical_value_eur
      FC Metaloglobus Bucharest    RO1            96000.0
                Boyacá Chicó FC   COL1           106000.0
Asociación Atlética Estudiantes   ARG1           127000.0
                Unión Magdalena   COL1           136000.0
                     Falkirk FC    SC1           137000.0

Clubs per league (top 10):
league
ARG1    30
MLS1    30
IT1     20
GB1     20
COL1    20
BRA1    20
ES1     20
JAP1    20
L1      18
FR1     18
Name: count, dtype: int64
